In [22]:
from typing import Dict, List

from model_ranking import (
    get_output_dir,
    get_summary_results,
    per_source_model_results,
    per_target_consis_result_type,
    per_target_performance_result_type,
    cmb_consistency_score_weighted_average,
    to_target_transfer_correlations,
    correlation_table,
)

In [2]:
output_dir = get_output_dir(
    source = "Hmito",
    target = "EPFL",
    model_name = "HmtoE_model4",
    output_folder = None,
    approach=None,
    result_type="predictions",
    base_seg_folder="/g/kreshuk/talks/model_ranking_results/AdaptiveBatchNorm/Mitochondria/train_mode_stats"
)
print(output_dir)

/g/kreshuk/talks/model_ranking_results/AdaptiveBatchNorm/Mitochondria/train_mode_stats/Hmito_to_EPFL_gap/HmtoE_model4/predictions


In [3]:
data_mapping = {
    "EPFL": "E",
    "Hmito": "H",
    "Rmito": "R",
    "VNC": "V",
}
consis_keys={
    "EPFL": "EI_consis",
    "Hmito": "EI_consis",
    "Rmito": "EI_consis",
    "VNC": "EI_consis",
}
consis_keys_bckg = {
    "EPFL": "EI_consis_bckg",
    "Hmito": "EI_consis_bckg",
    "Rmito": "EI_consis_bckg",
    "VNC": "EI_consis_bckg",
}

In [4]:
result_folders_StoT={
    "EPFL": "predictions",
    "Hmito": "predictions",
    "Rmito": "predictions",
    "VNC": "predictions",
}

per_target_norms_StoT: Dict[str, List[None]] = {
    "EPFL": [None],
    "Hmito": [None],
    "Rmito": [None],
    "VNC": [None],
}

perf_key_StoT = "F1_eval"
approach_StoT = None
base_result_path_StoT = "/g/kreshuk/talks/model_ranking_results/AdaptiveBatchNorm/Mitochondria/train_mode_stats"

In [5]:
result_folders_StoS={
    "EPFL": "P_full",
    "Hmito": "P_full",
    "Rmito": "P_full",
    "VNC": "P_full",
}

per_target_norms_StoS: Dict[str, List[str]] = {
    "EPFL": ["Normalize"],
    "Hmito": ["Normalize"],
    "Rmito": ["Normalize"],
    "VNC": ["Normalize"],
}

perf_key_StoS = "hard_f1"
approach_StoS = "consistency"
base_result_path_StoS = "/scratch/talks/consistency_results/patch_segmentation/mitochondria"

In [6]:
summary_results_postfix = "_full"
selected_augmentations = {
    "none": [""],
    "gauss": ["a007-01"],
}
consis_postfix="median"
perf_postfix="median"
targets = ["EPFL", "Hmito", "Rmito", "VNC"]
#targets = ["EPFL"]
sources = ["EPFL", "Hmito", "Rmito", "VNC"]
#sources = ["Hmito"]

In [7]:
model_set: Dict[str, List[str]] = {
    "EtoE": ["E_model5", "E_model_NA2", "E_model_Res1"],
    "EtoH": ["EtoHm_model5", "EtoHm_model_NA2", "EtoHm_model_Res1"],
    "EtoR": ["EtoRm_model5", "EtoRm_model_NA2", "EtoRm_model_Res1"],
    "EtoV": ["EtoV_model5", "EtoV_model_NA2", "EtoV_model_Res1"],
    "HtoE": ["HmtoE_model4", "HmtoE_model_NA2", "HmtoE_model_Res1"],
    "HtoH": ["Hm_model4", "Hm_model_NA2", "Hm_model_Res1"],
    "HtoR": ["HmtoRm_model4", "HmtoRm_model_NA2", "HmtoRm_model_Res1"],
    "HtoV": ["HmtoV_model4", "HmtoV_model_NA2", "HmtoV_model_Res1"],
    "RtoE": ["RmtoE_model4", "RmtoE_model_NA2", "RmtoE_model_Res1"],
    "RtoH": ["RmtoHm_model4", "RmtoHm_model_NA2", "RmtoHm_model_Res1"],
    "RtoR": ["Rm_model4", "Rm_model_NA2", "Rm_model_Res1"],
    "RtoV": ["RmtoV_model4", "RmtoV_model_NA2", "RmtoV_model_Res1"],
    "VtoE": ["VtoE_model2", "VtoE_model_NA2", "VtoE_model_Res1"],
    "VtoH": ["VtoHm_model2", "VtoHm_model_NA2", "VtoHm_model_Res1"],
    "VtoR": ["VtoRm_model2", "VtoRm_model_NA2", "VtoRm_model_Res1"],
}

In [8]:
per_target_forg_consistency: per_target_consis_result_type= {}
per_target_forg_performance: per_target_performance_result_type = {}
for i, target in enumerate(targets):
    per_source_forg_consistency = {}
    per_source_forg_performance = {}
    for source in sources:
        transfer = f"{data_mapping[source]}to{data_mapping[target]}"
        if transfer not in model_set:
            continue
        
        if source == target:
            base_result_path = base_result_path_StoS
            result_folders = result_folders_StoS
            per_target_norms = per_target_norms_StoS
            perf_key = perf_key_StoS
            approach = approach_StoS
        else:
            base_result_path = base_result_path_StoT
            result_folders = result_folders_StoT
            per_target_norms = per_target_norms_StoT
            perf_key = perf_key_StoT
            approach = approach_StoT

        for model in model_set[transfer]:
            consis_str, _, NA_perf = get_summary_results(
                source_data=[source],
                target_data=[target],
                selected_augmentations=selected_augmentations,
                consis_keys=consis_keys,
                result_folders=result_folders,
                source_models={source: model},
                selected_norms=per_target_norms,
                perf_key=perf_key,
                per_target_norms=True,
                approach=approach,
                consis_postfix=consis_postfix,
                perf_postfix=perf_postfix,
                summary_results_postfix=summary_results_postfix,
                base_seg_dir=base_result_path,
            )
            per_source_forg_consistency.update(
                per_source_model_results(
                    consis_str, 
                    source_models={source: model}, 
                ))
            per_source_forg_performance.update(
                per_source_model_results(
                    NA_perf, 
                    source_models={source: model},
                ))
    per_target_forg_consistency[target] = per_source_forg_consistency
    per_target_forg_performance[target] = per_source_forg_performance


Source: EPFL


100%|██████████| 1/1 [00:00<00:00, 63.13it/s]


Source: EPFL


100%|██████████| 1/1 [00:00<00:00, 88.42it/s]


Source: EPFL


...

100%|██████████| 1/1 [00:00<00:00, 127.03it/s]


Source: Rmito


100%|██████████| 1/1 [00:00<00:00, 110.88it/s]


Source: Rmito


100%|██████████| 1/1 [00:00<00:00, 124.15it/s]


In [9]:
per_target_bckg_consistency: per_target_consis_result_type= {}
per_target_bckg_performance: per_target_performance_result_type = {}
for i, target in enumerate(targets):
    per_source_bckg_consistency = {}
    per_source_bckg_performance = {}
    for source in sources:
        transfer = f"{data_mapping[source]}to{data_mapping[target]}"
        if transfer not in model_set:
            continue
        
        if source == target:
            base_result_path = base_result_path_StoS
            result_folders = result_folders_StoS
            per_target_norms = per_target_norms_StoS
            perf_key = perf_key_StoS
            approach = approach_StoS
        else:
            base_result_path = base_result_path_StoT
            result_folders = result_folders_StoT
            per_target_norms = per_target_norms_StoT
            perf_key = perf_key_StoT
            approach = approach_StoT

        for model in model_set[transfer]:
            consis_str, _, NA_perf = get_summary_results(
                source_data=[source],
                target_data=[target],
                selected_augmentations=selected_augmentations,
                consis_keys=consis_keys_bckg,
                result_folders=result_folders,
                source_models={source: model},
                selected_norms=per_target_norms,
                perf_key=perf_key,
                per_target_norms=True,
                approach=approach,
                consis_postfix=consis_postfix,
                perf_postfix=perf_postfix,
                summary_results_postfix=summary_results_postfix,
                base_seg_dir=base_result_path,
            )
            per_source_bckg_consistency.update(
                per_source_model_results(
                    consis_str, 
                    source_models={source: model}, 
                ))
            per_source_bckg_performance.update(
                per_source_model_results(
                    NA_perf, 
                    source_models={source: model},
                ))
    per_target_bckg_consistency[target] = per_source_bckg_consistency
    per_target_bckg_performance[target] = per_source_bckg_performance

Source: EPFL


100%|██████████| 1/1 [00:00<00:00, 163.98it/s]


Source: EPFL


100%|██████████| 1/1 [00:00<00:00, 167.48it/s]


Source: EPFL


...

100%|██████████| 1/1 [00:00<00:00, 165.75it/s]


Source: Rmito


100%|██████████| 1/1 [00:00<00:00, 166.10it/s]


Source: Rmito


100%|██████████| 1/1 [00:00<00:00, 165.89it/s]


In [10]:
per_target_cmb_consistency = cmb_consistency_score_weighted_average(
    per_target_forg_consistency,
    per_target_bckg_consistency,
    w_fg=0.5,
    w_bg=0.5,
    perturbation_key="gauss",
)

In [15]:
# per_target_consistency_a001_a003: Dict[str, Dict[str,float]] = {}
# per_target_consistency_a003_a005: Dict[str, Dict[str,float]] = {}
# per_target_consistency_a005_a007: Dict[str, Dict[str,float]] = {}
per_target_consistency_a007_a01: Dict[str, Dict[str,float]] = {}
#per_target_consistency_a01_a02: Dict[str, Dict[str,float]] = {}
for target, per_model_consistency in per_target_cmb_consistency.items():
    # per_target_consistency_a001_a003[target] = {}
    # per_target_consistency_a003_a005[target] = {}
    # per_target_consistency_a005_a007[target] = {}
    per_target_consistency_a007_a01[target] = {}
    #per_target_consistency_a01_a02[target] = {}
    for model, consistency in per_model_consistency.items():
        # per_target_consistency_a001_a003[target][model] = consistency['norm_Normalize']['gauss'][0]
        # per_target_consistency_a003_a005[target][model] = consistency['norm_Normalize']['gauss'][1]
        # per_target_consistency_a005_a007[target][model] = consistency['norm_Normalize']['gauss'][2]
        per_target_consistency_a007_a01[target][model] = consistency['norm_Normalize']['gauss'][0]
        #per_target_consistency_a01_a02[target][model] = consistency['norm_Normalize']['gauss'][4]


In [16]:
per_target_performance: Dict[str, Dict[str,float]] = {}
for target, per_model_performance in per_target_forg_performance.items():
    per_target_performance[target] = {}
    for model, performance in per_model_performance.items():
        if isinstance(performance, dict):
            per_target_performance[target][model] = performance['norm_Normalize']
        else:
            per_target_performance[target][model] = performance

In [ ]:
# import json
# from typing import Any
# import os
# from model_ranking import convert_numpy_types
# save_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/transfer_results/consistency"
# if not os.path.exists(save_path):
#     os.makedirs(save_path)
# with open(os.path.join(save_path, "transfer_AdaBN_train_GaussSweep_CMB_05f_05b_EI_scores_with_NORM.json"), "w") as f:
#     results: Dict[str, Any] = {
#         "transfer_scores": convert_numpy_types(per_target_cmb_consistency),
#     }
#     json.dump(results, f, indent=4)

In [ ]:
per_target_KT, per_target_SP, per_target_PE = to_target_transfer_correlations(
    targets,
    per_target_consistency_a007_a01,
    per_target_performance,
)

In [21]:
df = correlation_table(per_target_KT, per_target_SP, per_target_PE, targets)
print(df)

                kt  kt pval  s rho  s rho pval    pr  pr pval
Task targets                                                 
Mito EPFL     0.76     0.00   0.86        0.00  0.70     0.00
     Hmito    0.91     0.00   0.88        0.00  0.73     0.00
     Rmito    0.87     0.00   0.73        0.01  0.55     0.01
     VNC      0.71     0.03   0.47        0.26  0.39     0.17
